# Complete SecureRAG Pipeline with Multi-Round Retrieval (Ollama qwen3:0.6b)

This notebook runs an end-to-end SecureRAG workflow for a complicated question where multiple retrieval rounds improve answer quality.

## Scenario

You are advising leadership on a compound risk event that combines security policy drift, vendor concentration, and operations reliability regressions.

In [11]:
from __future__ import annotations

import atexit
import socket
import subprocess
import sys
import time
from pathlib import Path

import httpx

def _ensure_repo_on_path() -> Path:
    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        if (base / "securerag" / "__init__.py").exists():
            base_str = str(base)
            if base_str not in sys.path:
                sys.path.insert(0, base_str)
            return base
    raise RuntimeError(
        "Could not locate the SecureRAG repository root. Open this notebook from the secure-rag workspace."
    )

REPO_ROOT = _ensure_repo_on_path()
print("Using repository root:", REPO_ROOT)

from securerag import PrivacyConfig, PrivacyProtocol, SecureRAGAgent
from securerag.corpus import CorpusBuilder
from securerag.llm import OllamaLLM
from securerag.models import RawDocument

Using repository root: /Users/sonnguyen/research/secure-rag


In [12]:
def free_port() -> int:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind(("127.0.0.1", 0))
        return int(s.getsockname()[1])


def wait_for_health(base_url: str, proc: subprocess.Popen, timeout_s: float = 10.0) -> None:
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        if proc.poll() is not None:
            raise RuntimeError(
                "sim_server exited early. Ensure uvicorn is installed and the repository root is detected correctly."
            )
        try:
            r = httpx.get(f"{base_url}/health", timeout=0.5)
            if r.status_code == 200:
                return
        except Exception:
            pass
        time.sleep(0.1)
    raise RuntimeError(f"sim_server did not become healthy at {base_url}")


port = free_port()
base_url = f"http://127.0.0.1:{port}"
_env = dict(**__import__("os").environ)
_existing_pp = _env.get("PYTHONPATH", "")
_env["PYTHONPATH"] = str(REPO_ROOT) if not _existing_pp else f"{str(REPO_ROOT)}:{_existing_pp}"

proc = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "uvicorn",
        "securerag.sim_server:app",
        "--host",
        "127.0.0.1",
        "--port",
        str(port),
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    env=_env,
    cwd=str(REPO_ROOT),
)
wait_for_health(base_url, proc)
print("Backend ready at", base_url)


def cleanup() -> None:
    if proc.poll() is None:
        proc.terminate()


_ = atexit.register(cleanup)

Backend ready at http://127.0.0.1:50199


In [13]:
documents = [
    RawDocument(doc_id="board-1", text="Board memo: incident recurrence increased by 27% after deferred remediation from Q2 into Q3."),
    RawDocument(doc_id="board-2", text="Board memo: top two vendors now serve 68% of critical workflows, increasing concentration risk."),
    RawDocument(doc_id="audit-iam", text="IAM audit: dormant privileged accounts remained active beyond policy limits."),
    RawDocument(doc_id="audit-patch", text="Patch governance report: SLA misses correlate with weekend staffing gaps."),
    RawDocument(doc_id="ops-queues", text="Ops telemetry: queue depth spikes precede customer-visible timeout bursts by 10-20 minutes."),
    RawDocument(doc_id="ops-change", text="Change logs: emergency deployments bypassed standard review in two high-impact incidents."),
    RawDocument(doc_id="policy", text="Security policy: all critical findings must have owner, due date, and escalation path."),
    RawDocument(doc_id="vendor", text="Vendor oversight note: concentration plus delayed remediations creates compounding outage and compliance risk."),
]

corpus = (
    CorpusBuilder(PrivacyProtocol.DIFF_PRIVACY, backend_url=base_url)
    .with_chunk_size(180)
    .with_overlap(40)
    .add_documents(documents)
    .build()
)

config = PrivacyConfig(
    protocol=PrivacyProtocol.DIFF_PRIVACY,
    backend=base_url,
    epsilon=250.0,
    noise_std=0.22,
    top_k=4,
    max_rounds=6,
)

llm = OllamaLLM(model="qwen3:0.6b", use_ollama=True, timeout_s=60.0)
agent = SecureRAGAgent.from_config(config, corpus, llm=llm)

In [14]:
complex_question = (
    "Using policy, audit, and operations evidence together, explain the most likely causal chain behind the Q3 reliability and compliance deterioration. "
    "Then recommend a phased mitigation plan with immediate and 90-day actions."
)

result = agent.run(complex_question)

print("=== Final Answer ===")
print(result.answer)
print()
print("=== Retrieval Diagnostics ===")
print("Rounds used:", result.rounds)
print("Context size:", result.context_size)
print("Budget snapshot:", agent.budget_snapshot())

=== Final Answer ===
The most likely causal chain is: **policy controls (security policies) are in place, but when these policies are not enforced (due to delayed remediation), leading to compromised security. The vendor's oversight note highlights that concentration of issues (like delayed remediation) is causing compounding outages and compliance risks. The IAM audit shows that dormant accounts remain active beyond policy limits, which is a policy violation. The board memo indicates that incident recurrence increased after deferring remediation, showing unresolved issues. The ops telemetry shows that system stress (due to queue depth spikes) precedes timeout bursts, indicating underlying system overload.**  

**Phased mitigation plan:**  
**Immediate actions:**  
1. Verify remediation status and ensure all policy requirements are met.  
2. Address any dormant accounts by terminating or reassigning privileges.  
3. Monitor and resolve queue depth spikes before timeout bursts.  

**90-

In [15]:
# Optional: compare single-round retrieval quality to multi-round agent output.
one_round_docs = agent.retriever.retrieve(complex_question, round_n=0)
print("Top docs from one round:")
for d in one_round_docs[:4]:
    print(f"- {d.doc_id} (score={d.score:.3f})")

Top docs from one round:
- policy (score=0.147)
- vendor (score=0.109)
- audit-iam (score=0.095)
- board-1 (score=0.072)


In [7]:
if proc.poll() is None:
    proc.terminate()
print("Stopped local sim backend")

Stopped local sim backend


In [16]:
# Compact diagnostics to verify result quality without huge output dumps.
snap = agent.budget_snapshot()
print({
    "rounds": result.rounds,
    "context_size": result.context_size,
    "one_round_count": len(one_round_docs),
    "top_one_round_doc_ids": [d.doc_id for d in one_round_docs[:4]],
    "budget_spent": round(float(snap.get("spent", 0.0)), 6),
    "budget_remaining": round(float(snap.get("remaining", 0.0)), 6),
    "ledger_entries": len(snap.get("ledger", [])),
})

preview = result.answer[:500].replace("\n", " ")
print("answer_preview:", preview)

{'rounds': 3, 'context_size': 5, 'one_round_count': 4, 'top_one_round_doc_ids': ['policy', 'vendor', 'audit-iam', 'board-1'], 'budget_spent': 73.496397, 'budget_remaining': 176.874988, 'ledger_entries': 3}
answer_preview: The most likely causal chain is: **policy controls (security policies) are in place, but when these policies are not enforced (due to delayed remediation), leading to compromised security. The vendor's oversight note highlights that concentration of issues (like delayed remediation) is causing compounding outages and compliance risks. The IAM audit shows that dormant accounts remain active beyond policy limits, which is a policy violation. The board memo indicates that incident recurrence increa
